In [ ]:
# 1. SETUP & MOUNT
from google.colab import drive
import os

drive.mount('/content/drive')
!pip install rapidfuzz lightgbm joblib>=1.4.0 cupy-cuda12x

import shutil
from pathlib import Path

DRIVE_DATA = Path('/content/drive/MyDrive/Amazon-ML-entity-resolution/dataset')
LOCAL_DATA = Path('/content/dataset')

if DRIVE_DATA.exists():
    print("Copying dataset from Drive to local NVMe SSD (takes ~1-2 mins)...")
    shutil.copytree(DRIVE_DATA, LOCAL_DATA, dirs_exist_ok=True)
    print("✅ Dataset ready in local workspace.")
else:
    print("Dataset not found in Drive. Make sure the path is correct.")
    
# Create output directories
for d in ['/content/output', '/content/models', '/content/experiments', '/content/reports/error_analysis']:
    os.makedirs(d, exist_ok=True)


In [ ]:
# --- config.py ---
"""
config.py — Central configuration for the entity resolution pipeline.
All paths, hyperparameters, and constants live here.
"""
from pathlib import Path

# ── Project layout ────────────────────────────────────────────────────────────
ROOT = Path("/content")\n
TRAIN_DIR  = ROOT / "dataset" / "train"
TEST_DIR   = ROOT / "dataset" / "test"
OUTPUT_DIR = ROOT / "output"
MODEL_DIR  = ROOT / "models"
SRC_DIR    = ROOT / "src"

for _d in (OUTPUT_DIR, MODEL_DIR):
    _d.mkdir(exist_ok=True)

# ── Blocking ──────────────────────────────────────────────────────────────────
# TF-IDF vectoriser (character n-grams, l2-normalised → dot == cosine)
TFIDF_ANALYZER     = "char_wb"   # pad with word-boundary spaces
TFIDF_NGRAM_RANGE  = (2, 4)      # 2- to 4-character n-grams
TFIDF_MAX_FEATURES = 50_000      # vocabulary cap
TFIDF_MIN_DF       = 2           # ignore singletons

# Candidate retrieval
# K-sweep results (5K dev run):
#   K=10 → recall=86.1%, F₀.₅=0.9238  K=30 → recall=88.7%, F₀.₅=0.9411
#   K=50 → recall=89.5%, F₀.₅=0.9472  K=100 → recall=90.2%, F₀.₅=0.9534
# Recommendation: K=50 for 50K run, K=100 for full training.
# Bottleneck is blocking recall (AUC=1.0 at all K), not the model.
K_CANDIDATES   = 50   # top-K per (S1, source) pair  → up to 100 per S1
MIN_SIM_SCORE  = 0.05 # discard near-zero cosine hits

# Memory-safe batching during cosine search
S1_BATCH_SIZE  = 300   # S1 rows per iteration
S23_CHUNK_SIZE = 400_000  # S23 rows per chunk (controls peak RAM)

# ── Train / validation split ──────────────────────────────────────────────────
VAL_FRACTION = 0.20
RANDOM_STATE = 42

# Negative sampling for classifier training
NEG_PER_POS = 5   # hard negatives drawn from blocking candidates

# ── LightGBM ──────────────────────────────────────────────────────────────────
LGBM_PARAMS = dict(
    objective        = "binary",
    metric           = "auc",
    num_leaves       = 127,
    learning_rate    = 0.05,
    n_estimators     = 1000,
    min_child_samples= 20,
    subsample        = 0.80,
    colsample_bytree = 0.80,
    reg_alpha        = 0.1,
    reg_lambda       = 0.1,
    n_jobs           = -1,
    random_state     = RANDOM_STATE,
    verbose          = -1,
)
EARLY_STOPPING_ROUNDS = 50

# Threshold grid — sweep on validation F₀.₅
THRESHOLD_GRID = [round(t, 2) for t in [x * 0.05 for x in range(4, 20)]]
# [0.20, 0.25, 0.30, ..., 0.95]


In [ ]:
# --- preprocess.py ---
"""
preprocess.py — Text normalization for business names and addresses.

Key operations
--------------
1. Unicode NFKC normalization  (handles Devanagari, Kannada, accented chars)
2. Lowercase + punctuation removal
3. Abbreviation expansion      (Corp→Corporation, Rd→Road, etc.)
4. Whitespace collapsing
5. Numeric token extraction    (for address number overlap feature)
"""

import re
import unicodedata

import pandas as pd

# ── Abbreviation lookup tables ────────────────────────────────────────────────
# Compile once at module load for speed.

_NAME_ABBREVS_RAW = {
    # Legal suffixes
    r"\bcorp\b":        "corporation",
    r"\bco\b":          "company",
    r"\binc\b":         "incorporated",
    r"\bltd\b":         "limited",
    r"\bllc\b":         "limited liability company",
    r"\bllp\b":         "limited liability partnership",
    r"\bpvt\b":         "private",
    r"\bpvt ltd\b":     "private limited",
    r"\bpte\b":         "private",
    r"\bpte ltd\b":     "private limited",
    r"\blp\b":          "limited partnership",
    r"\bplc\b":         "public limited company",
    r"\bsas\b":         "societe par actions simplifiee",  # French
    r"\bsarl\b":        "societe a responsabilite limitee",
    r"\bsa\b":          "societe anonyme",
    # Common business words
    r"\b&\b":           "and",
    r"\bsvc\b":         "services",
    r"\bsvcs\b":        "services",
    r"\btech\b":        "technology",
    r"\btechnol\b":     "technology",
    r"\bintl\b":        "international",
    r"\bmfg\b":         "manufacturing",
    r"\bmgmt\b":        "management",
    r"\bassoc\b":       "associates",
    r"\bassn\b":        "association",
    r"\bnatl\b":        "national",
    r"\bgrp\b":         "group",
    r"\bent\b":         "enterprises",
    r"\bhldgs\b":       "holdings",
    r"\binds\b":        "industries",
    r"\bdist\b":        "distributors",
    r"\bdistrib\b":     "distributors",
    r"\bsol\b":         "solutions",
    r"\bsoln\b":        "solutions",
    r"\bsolns\b":       "solutions",
    r"\bmkt\b":         "marketing",
    r"\bmktg\b":        "marketing",
    r"\badv\b":         "advertising",
    r"\bconst\b":       "construction",
    r"\bconsult\b":     "consulting",
    r"\bdev\b":         "development",
    r"\bfinl\b":        "financial",
    r"\bfin\b":         "finance",
    r"\binsur\b":       "insurance",
    r"\bhosp\b":        "hospital",
    r"\bmedl\b":        "medical",
    r"\bmed\b":         "medical",
    r"\bpharm\b":       "pharmaceutical",
    r"\brealty\b":      "realty",
    r"\brestnt\b":      "restaurant",
    r"\brest\b":        "restaurant",
    r"\brestaurant\b":  "restaurant",
}

_ADDR_ABBREVS_RAW = {
    # Street types
    r"\brd\b":      "road",
    r"\bst\b":      "street",
    r"\bave\b":     "avenue",
    r"\bav\b":      "avenue",
    r"\bblvd\b":    "boulevard",
    r"\bdr\b":      "drive",
    r"\bln\b":      "lane",
    r"\bct\b":      "court",
    r"\bpl\b":      "place",
    r"\bsq\b":      "square",
    r"\bhwy\b":     "highway",
    r"\bfwy\b":     "freeway",
    r"\bpkwy\b":    "parkway",
    r"\bexpy\b":    "expressway",
    r"\bxing\b":    "crossing",
    r"\btrl\b":     "trail",
    r"\bcir\b":     "circle",
    # Unit designators
    r"\bste\b":     "suite",
    r"\bapt\b":     "apartment",
    r"\bflr\b":     "floor",
    r"\bfl\b":      "floor",
    r"\bbldg\b":    "building",
    r"\bunit\b":    "unit",
    # Directions
    r"\bnorth\b":   "north",
    r"\bsouth\b":   "south",
    r"\beast\b":    "east",
    r"\bwest\b":    "west",
    r"\bnw\b":      "northwest",
    r"\bne\b":      "northeast",
    r"\bsw\b":      "southwest",
    r"\bse\b":      "southeast",
}


def _compile_abbrevs(raw: dict) -> list:
    return [(re.compile(pat, re.IGNORECASE), repl) for pat, repl in raw.items()]


_NAME_ABBREVS  = _compile_abbrevs(_NAME_ABBREVS_RAW)
_ADDR_ABBREVS  = _compile_abbrevs(_ADDR_ABBREVS_RAW)

_PUNCT_RE      = re.compile(r"[^\w\s]")
_SPACE_RE      = re.compile(r"\s+")
_DIGIT_RE      = re.compile(r"\d+")


# ── Core helpers ──────────────────────────────────────────────────────────────

def _safe_str(value) -> str:
    """Convert any value to string, returning '' for NaN/None/'nan'."""
    if value is None:
        return ""
    if isinstance(value, float):
        import math
        if math.isnan(value):
            return ""
        return str(value)
    s = str(value).strip()
    # Catch the string literal 'nan' / 'NaN' / 'NAN' (from dtype=str reads)
    if s.lower() == "nan":
        return ""
    return s


def _unicode_norm(text: str) -> str:
    """NFKC normalization: decomposes compatibility characters, reassembles."""
    return unicodedata.normalize("NFKC", text)


def _apply_abbrevs(text: str, abbrevs: list) -> str:
    for pattern, replacement in abbrevs:
        text = pattern.sub(replacement, text)
    return text


def normalize_name(text, expand_abbrevs: bool = True) -> str:
    """
    Full normalization pipeline for business names.

    >>> normalize_name("ABC Corp. & Associates Ltd.")
    'abc corporation and associates limited'
    """
    text = _unicode_norm(_safe_str(text))
    text = text.lower()
    text = _PUNCT_RE.sub(" ", text)
    text = _SPACE_RE.sub(" ", text).strip()
    if expand_abbrevs:
        text = _apply_abbrevs(text, _NAME_ABBREVS)
        text = _SPACE_RE.sub(" ", text).strip()
    return text


def normalize_address(text, expand_abbrevs: bool = True) -> str:
    """
    Full normalization pipeline for addresses.

    >>> normalize_address("14 Main Rd, Apt 2B, Near SBI ATM")
    '14 main road apartment 2b near sbi atm'
    """
    text = _unicode_norm(_safe_str(text))
    text = text.lower()
    text = _PUNCT_RE.sub(" ", text)
    text = _SPACE_RE.sub(" ", text).strip()
    if expand_abbrevs:
        text = _apply_abbrevs(text, _ADDR_ABBREVS)
        text = _SPACE_RE.sub(" ", text).strip()
    return text


def extract_numbers(text) -> list[str]:
    """Return list of digit sequences found in text (for number-overlap feature)."""
    return _DIGIT_RE.findall(_safe_str(text))


# ── DataFrame helper ──────────────────────────────────────────────────────────

def preprocess_df(df: pd.DataFrame) -> pd.DataFrame:
    """
    Add derived columns to a source TSV DataFrame.

    New columns:
        name_norm   — normalized business name
        addr_norm   — normalized business address
        country_norm— lowercased country string
        addr_nums   — space-joined numeric tokens from raw address
        blocking_text — name_norm used as the TF-IDF input for blocking
    """
    df = df.copy()
    # Explicit fillna — guards against both actual NaN floats and the
    # string literal 'nan' that appears when TSVs are read with dtype=str.
    for col in ("business_name", "business_address", "country"):
        if col in df.columns:
            df[col] = df[col].fillna("").astype(str).str.strip()
            df.loc[df[col].str.lower() == "nan", col] = ""
    df["name_norm"]  = df["business_name"].apply(normalize_name)
    df["addr_norm"]  = df["business_address"].apply(normalize_address)
    df["country_norm"] = df["country"].apply(
        lambda x: _safe_str(x).lower().strip()
    )
    df["addr_nums"]  = df["business_address"].apply(
        lambda x: " ".join(extract_numbers(x))
    )
    # Blocking text: name + partial country (helps cross-country confusion)
    df["blocking_text"] = df["name_norm"]
    return df


In [ ]:
# --- blocking_multi.py ---
"""
blocking_multi.py — Multi-Blocker Candidate Generation Suite.

Implements and evaluates:
1. Blocker A: Name-only TF-IDF Cosine (Baseline)
2. Blocker B: Address-only TF-IDF Cosine
3. Blocker C: Combined Name+Address TF-IDF Cosine
4. Blocker D: Exact Normalized Name Inverted Map
5. Blocker E: Rare Token Inverted Index
6. Blocker F: Numeric / House Number Overlap Index

Merges candidate pools via deduplicated score/frequency union.
Reports Candidate Recall, Avg Candidates / S1, Missed Matches Recovered, and Runtime.
"""

import gc
import re
import time
import logging
from collections import defaultdict
from pathlib import Path
from typing import Dict, List, Tuple, Set

import numpy as np
import pandas as pd
import scipy.sparse as sp
from sklearn.feature_extraction.text import TfidfVectorizer

    K_CANDIDATES,
    MIN_SIM_SCORE,
    S1_BATCH_SIZE,
    S23_CHUNK_SIZE,
    TFIDF_ANALYZER,
    TFIDF_MAX_FEATURES,
    TFIDF_MIN_DF,
    TFIDF_NGRAM_RANGE,
)

logger = logging.getLogger(__name__)


# ── TF-IDF Helper ─────────────────────────────────────────────────────────────

def _build_vectorizer(texts: List[str]) -> Tuple[TfidfVectorizer, sp.csr_matrix]:
    vec = TfidfVectorizer(
        analyzer=TFIDF_ANALYZER,
        ngram_range=TFIDF_NGRAM_RANGE,
        max_features=TFIDF_MAX_FEATURES,
        min_df=TFIDF_MIN_DF,
        sublinear_tf=True,
        strip_accents=None,
        norm="l2",
        dtype=np.float32,
    )
    matrix = vec.fit_transform(texts)
    return vec, matrix


def _batched_topk(
    s1_mat: sp.csr_matrix,
    s23_mat: sp.csr_matrix,
    s23_ids: np.ndarray,
    k: int,
    min_score: float = MIN_SIM_SCORE,
) -> List[List[Tuple[str, float]]]:
    n1  = s1_mat.shape[0]
    n23 = s23_mat.shape[0]
    k   = min(k, n23)

    results: List[List[Tuple[str, float]]] = [[] for _ in range(n1)]
    top_ids    = [np.empty(0, dtype=object)   for _ in range(n1)]
    top_scores = [np.empty(0, dtype=np.float32) for _ in range(n1)]

    s1_batch  = S1_BATCH_SIZE
    s23_chunk = S23_CHUNK_SIZE

    try:
        import cupy as cp
        from cupyx.scipy import sparse as cusparse
        HAS_CUPY = True
    except ImportError:
        HAS_CUPY = False

    for s23_start in range(0, n23, s23_chunk):
        s23_end   = min(s23_start + s23_chunk, n23)
        chunk_mat = s23_mat[s23_start:s23_end]
        chunk_ids = s23_ids[s23_start:s23_end]

        if HAS_CUPY:
            chunk_mat_gpu = cusparse.csr_matrix(chunk_mat)

        for s1_start in range(0, n1, s1_batch):
            s1_end = min(s1_start + s1_batch, n1)
            q_mat  = s1_mat[s1_start:s1_end]

            if HAS_CUPY:
                q_mat_dense_gpu = cp.array(q_mat.toarray(), dtype=np.float32)
                sim_block_gpu = chunk_mat_gpu.dot(q_mat_dense_gpu.T).T
                sim_block = cp.asnumpy(sim_block_gpu)
            else:
                sim_block = q_mat.dot(chunk_mat.T)
                if sp.issparse(sim_block):
                    sim_block = sim_block.toarray()
                sim_block = sim_block.astype(np.float32)

            for local_i in range(s1_end - s1_start):
                global_i = s1_start + local_i
                row = sim_block[local_i]
                mask = row >= min_score
                if not mask.any():
                    continue

                new_ids    = chunk_ids[mask]
                new_scores = row[mask]

                merged_ids    = np.concatenate([top_ids[global_i],    new_ids])
                merged_scores = np.concatenate([top_scores[global_i], new_scores])

                if len(merged_ids) > k:
                    idx = np.argpartition(merged_scores, -k)[-k:]
                else:
                    idx = np.arange(len(merged_ids))

                top_ids[global_i]    = merged_ids[idx]
                top_scores[global_i] = merged_scores[idx]

    for i in range(n1):
        if len(top_ids[i]) == 0:
            results[i] = []
            continue
        order = np.argsort(top_scores[i])[::-1]
        results[i] = [(top_ids[i][j], float(top_scores[i][j])) for j in order]
    return results


# ── 1. TF-IDF Cosine Blocker (Name or Address or Combined) ────────────────────

def block_tfidf(
    s1_df: pd.DataFrame,
    s23_df: pd.DataFrame,
    text_column: str = "name_norm",
    k: int = 30,
) -> Dict[str, List[Tuple[str, float]]]:
    """Country-bucketed TF-IDF cosine blocker on specified text column."""
    all_cands: Dict[str, List[Tuple[str, float]]] = {eid: [] for eid in s1_df["entity_id"]}

    common_countries = set(s1_df["country_norm"].unique()) & set(s23_df["country_norm"].unique())

    for ctry in sorted(common_countries):
        s1_sub  = s1_df[s1_df["country_norm"] == ctry]
        s23_sub = s23_df[s23_df["country_norm"] == ctry]
        if s1_sub.empty or s23_sub.empty:
            continue

        vec, s23_mat = _build_vectorizer(s23_sub[text_column].tolist())
        s1_vecs      = vec.transform(s1_sub[text_column].tolist())
        s23_ids      = s23_sub["entity_id"].values

        hits = _batched_topk(s1_vecs, s23_mat, s23_ids, k=k)
        for sid, hit_list in zip(s1_sub["entity_id"], hits):
            all_cands[sid].extend(hit_list)

        del vec, s23_mat, s1_vecs
        gc.collect()

    return all_cands


# ── 2. Exact Normalized Name Inverted Index ────────────────────────────────────

def block_exact_name(
    s1_df: pd.DataFrame,
    s23_df: pd.DataFrame,
    max_matches_per_name: int = 50,
) -> Dict[str, List[Tuple[str, float]]]:
    """Exact normalized name hashtable lookup (O(1) dictionary retrieval)."""
    # Build hashtable: {country_norm: {name_norm: [s23_id, ...]}}
    table = defaultdict(lambda: defaultdict(list))
    for _, row in s23_df.iterrows():
        ctry = row["country_norm"]
        name = row["name_norm"]
        if name:
            table[ctry][name].append(row["entity_id"])

    all_cands = {}
    for _, row in s1_df.iterrows():
        sid  = row["entity_id"]
        ctry = row["country_norm"]
        name = row["name_norm"]
        
        matches = table[ctry].get(name, [])
        if matches:
            # Assign fixed high score 1.0 for exact name match
            all_cands[sid] = [(m, 1.0) for m in matches[:max_matches_per_name]]
        else:
            all_cands[sid] = []

    return all_cands


# ── 3. Rare Token Inverted Index ───────────────────────────────────────────────

def block_rare_tokens(
    s1_df: pd.DataFrame,
    s23_df: pd.DataFrame,
    max_token_freq: int = 30,
    max_cands_per_s1: int = 15,
) -> Dict[str, List[Tuple[str, float]]]:
    """Inverted index matching on rare business name/address tokens."""
    # Build document frequency of tokens per country
    df_count = defaultdict(lambda: defaultdict(int))
    token_to_s23 = defaultdict(lambda: defaultdict(list))

    for _, row in s23_df.iterrows():
        ctry = row["country_norm"]
        sid  = row["entity_id"]
        tokens = set((row["name_norm"] + " " + row["addr_norm"]).split())
        for t in tokens:
            if len(t) >= 4:  # ignore short generic words
                df_count[ctry][t] += 1
                token_to_s23[ctry][t].append(sid)

    all_cands = {}
    for _, row in s1_df.iterrows():
        sid  = row["entity_id"]
        ctry = row["country_norm"]
        tokens = set((row["name_norm"] + " " + row["addr_norm"]).split())

        hit_counts = defaultdict(int)
        for t in tokens:
            if len(t) >= 4 and df_count[ctry].get(t, 999999) <= max_token_freq:
                for target_id in token_to_s23[ctry][t]:
                    hit_counts[target_id] += 1

        if hit_counts:
            # Sort by number of rare token overlaps
            sorted_hits = sorted(hit_counts.items(), key=lambda x: -x[1])[:max_cands_per_s1]
            all_cands[sid] = [(tid, float(count) * 0.2) for tid, count in sorted_hits]
        else:
            all_cands[sid] = []

    return all_cands


# ── 4. Numeric / Building Number Index ─────────────────────────────────────────

def block_numeric_address(
    s1_df: pd.DataFrame,
    s23_df: pd.DataFrame,
    max_cands_per_s1: int = 15,
) -> Dict[str, List[Tuple[str, float]]]:
    """Inverted index matching on address numeric sequences (building numbers/PINs)."""
    table = defaultdict(lambda: defaultdict(list))
    for _, row in s23_df.iterrows():
        ctry = row["country_norm"]
        sid  = row["entity_id"]
        nums = set(extract_numbers(row["business_address"]))
        for n in nums:
            if len(n) >= 2:  # ignore single digits
                table[ctry][n].append(sid)

    all_cands = {}
    for _, row in s1_df.iterrows():
        sid  = row["entity_id"]
        ctry = row["country_norm"]
        nums = set(extract_numbers(row["business_address"]))

        hit_counts = defaultdict(int)
        for n in nums:
            if len(n) >= 2 and len(table[ctry].get(n, [])) <= 100:
                for target_id in table[ctry][n]:
                    hit_counts[target_id] += 1

        if hit_counts:
            sorted_hits = sorted(hit_counts.items(), key=lambda x: -x[1])[:max_cands_per_s1]
            all_cands[sid] = [(tid, float(count) * 0.15) for tid, count in sorted_hits]
        else:
            all_cands[sid] = []

    return all_cands


# ── Multi-Blocker Union Strategy ───────────────────────────────────────────────

def generate_multi_blocker_candidates(
    s1_df: pd.DataFrame,
    s2_df: pd.DataFrame,
    s3_df: pd.DataFrame,
    blocker_configs: List[dict],
    k_final: int = K_CANDIDATES,
) -> Tuple[Dict[str, List[str]], Dict[str, dict]]:
    """
    Executes multiple blockers and merges candidate pools.

    Returns:
      final_candidates: {s1_id: [candidate_s23_ids]}
      diagnostics: breakdown of candidates recovered per blocker
    """
    s23_df = pd.concat([s2_df, s3_df], ignore_index=True)
    all_s1_ids = s1_df["entity_id"].tolist()

    blocker_results = {}
    blocker_runtimes = {}

    for cfg in blocker_configs:
        b_type = cfg["type"]
        b_name = cfg["name"]
        t0 = time.time()
        logger.info(f"Running Blocker [{b_name}] (type={b_type})...")

        if b_type == "name_tfidf":
            res = block_tfidf(s1_df, s23_df, text_column="name_norm", k=cfg.get("k", 30))
        elif b_type == "addr_tfidf":
            res = block_tfidf(s1_df, s23_df, text_column="addr_norm", k=cfg.get("k", 20))
        elif b_type == "combo_tfidf":
            s1_sub = s1_df.copy()
            s23_sub = s23_df.copy()
            s1_sub["combo"] = s1_sub["name_norm"] + " " + s1_sub["addr_norm"]
            s23_sub["combo"] = s23_sub["name_norm"] + " " + s23_sub["addr_norm"]
            res = block_tfidf(s1_sub, s23_sub, text_column="combo", k=cfg.get("k", 20))
        elif b_type == "exact_name":
            res = block_exact_name(s1_df, s23_df, max_matches_per_name=cfg.get("max_matches", 30))
        elif b_type == "rare_tokens":
            res = block_rare_tokens(s1_df, s23_df, max_cands_per_s1=cfg.get("max_cands", 15))
        elif b_type == "numeric_addr":
            res = block_numeric_address(s1_df, s23_df, max_cands_per_s1=cfg.get("max_cands", 15))
        else:
            raise ValueError(f"Unknown blocker type: {b_type}")

        elapsed = time.time() - t0
        blocker_results[b_name] = res
        blocker_runtimes[b_name] = round(elapsed, 2)
        logger.info(f"  Finished Blocker [{b_name}] in {elapsed:.2f}s")

    # Merge candidates for each S1 entity
    final_candidates = {}
    recovery_stats = {b_name: 0 for b_name in blocker_results}

    for sid in all_s1_ids:
        score_map = defaultdict(float)
        blocker_origins = defaultdict(set)

        for b_name, b_dict in blocker_results.items():
            hits = b_dict.get(sid, [])
            for cid, score in hits:
                if score > score_map[cid]:
                    score_map[cid] = score
                blocker_origins[cid].add(b_name)

        sorted_cands = sorted(score_map.items(), key=lambda x: -x[1])[:k_final]
        final_cands_list = [cid for cid, _ in sorted_cands]
        final_candidates[sid] = final_cands_list

        for cid in final_cands_list:
            for b_name in blocker_origins[cid]:
                recovery_stats[b_name] += 1

    diagnostics = {
        "runtimes": blocker_runtimes,
        "contributions": recovery_stats,
    }

    return final_candidates, diagnostics


In [ ]:
# --- blocking.py ---
"""
blocking.py — Multi-Blocker Candidate Generation Pipeline.

Combines:
1. Name-TF-IDF cosine blocker (char 2-4 n-grams, K=30)
2. Address-TF-IDF cosine blocker (char 2-4 n-grams, K=15)
3. Exact normalized name hashtable blocker (max 15 matches)
4. Rare-token inverted index blocker (max 10 matches)

Achieves 99.25% candidate recall with ~45.7 candidates per S1 entity.
"""

import logging
import pandas as pd
from pathlib import Path
from typing import Dict, List

logger = logging.getLogger(__name__)

DEFAULT_BLOCKER_CONFIGS = [
    {"name": "name_tfidf", "type": "name_tfidf", "k": 30},
    {"name": "addr_tfidf", "type": "addr_tfidf", "k": 15},
    {"name": "exact_name", "type": "exact_name", "max_matches": 15},
    {"name": "rare_tokens", "type": "rare_tokens", "max_cands": 10},
]


def generate_candidates(
    s1_df: pd.DataFrame,
    s2_df: pd.DataFrame,
    s3_df: pd.DataFrame,
    k: int = 50,
) -> Dict[str, List[str]]:
    """
    Multi-blocker candidate generation pipeline.
    Returns {s1_entity_id: [candidate_s23_entity_ids]} sorted by combined relevance.
    """
    logger.info("Executing Multi-Blocker Candidate Union Pipeline...")
    cands, diag = generate_multi_blocker_candidates(
        s1_df, s2_df, s3_df,
        blocker_configs=DEFAULT_BLOCKER_CONFIGS,
        k_final=k,
    )
    return cands


def save_candidate_pairs(
    candidates: Dict[str, List[str]],
    s1_ids: List[str],
    output_path: Path,
) -> None:
    rows = [
        {
            "source1_entity_id":    s1_id,
            "candidate_entity_ids": ",".join(candidates.get(s1_id, [])),
        }
        for s1_id in s1_ids
    ]
    pd.DataFrame(rows).to_csv(output_path, sep="\t", index=False)
    n_with = sum(1 for r in rows if r["candidate_entity_ids"])
    logger.info(
        f"Saved {len(rows)} rows → {output_path}  "
        f"({n_with} with candidates, {len(rows)-n_with} singletons)"
    )


In [ ]:
# --- features.py ---
"""
features.py — Pair-level similarity & domain interaction features for LightGBM.

Computes 28 similarity, domain interaction, digit conflict, and structural features per pair:

Base Similarity Features (17)
-----------------------------
1. name_ratio          — Levenshtein ratio (rapidfuzz)
2. name_token_sort     — token-sort ratio   (handles word reorder)
3. name_token_set      — token-set ratio    (handles subset / extra words)
4. name_partial        — partial-string ratio (for abbreviations)
5. name_jaro_winkler   — Jaro-Winkler similarity
6. name_jaccard_token  — Jaccard of word-token sets
7. addr_ratio          — Levenshtein ratio
8. addr_token_sort     — token-sort ratio
9. addr_token_set      — token-set ratio
10. addr_jaccard_token — Jaccard of word-token sets
11. addr_num_overlap   — Jaccard of numeric-token sets (building/pin numbers)
12. country_match      — 1 if country_norm identical, 0 otherwise
13. blocking_score     — cosine similarity from TF-IDF blocking step
14. name_len_s1        — len(name_norm) of S1 entity
15. name_len_s23       — len(name_norm) of S23 entity
16. name_len_ratio     — min/max of name lengths (0-1)
17. addr_len_ratio     — min/max of addr_norm lengths (0-1)

Advanced Interaction & Domain Features (11)
--------------------------------------------
18. name_x_address              — name_jw * addr_token_set
19. min_name_address            — min(name_token_set, addr_token_set)
20. max_name_address            — max(name_token_set, addr_token_set)
21. has_both_addresses          — 1 if both S1 and S23 addresses >= 5 chars
22. missing_address_asymmetry   — 1 if one address >= 10 chars while other < 3 chars
23. exact_name_match            — 1 if normalized names are identical
24. conflicting_digits          — 1 if both have numbers but d1 ∩ d2 is empty
25. strong_name_and_number_match — 1 if name_token_set >= 0.85 and addr_num_overlap >= 0.5
26. strong_name_but_number_conflict — 1 if name_token_set >= 0.85 and conflicting_digits == 1
27. candidate_is_s3             — 1 if candidate starts with S3-
28. candidate_rank              — 1-based rank of candidate within S1 candidate list

Total: 28 features.
"""

import logging
import re
from typing import Dict, List, Tuple, Set

import numpy as np
import pandas as pd

try:
    from rapidfuzz import fuzz as rfuzz
    from rapidfuzz.distance import JaroWinkler
    _HAS_RAPIDFUZZ = True
except ImportError:
    _HAS_RAPIDFUZZ = False
    import difflib

logger = logging.getLogger(__name__)

FEATURE_NAMES = [
    "name_ratio",
    "name_token_sort",
    "name_token_set",
    "name_partial",
    "name_jaro_winkler",
    "name_jaccard_token",
    "addr_ratio",
    "addr_token_sort",
    "addr_token_set",
    "addr_jaccard_token",
    "addr_num_overlap",
    "country_match",
    "blocking_score",
    "name_len_s1",
    "name_len_s23",
    "name_len_ratio",
    "addr_len_ratio",
    # Advanced 11
    "name_x_address",
    "min_name_address",
    "max_name_address",
    "has_both_addresses",
    "missing_address_asymmetry",
    "exact_name_match",
    "conflicting_digits",
    "strong_name_and_number_match",
    "strong_name_but_number_conflict",
    "candidate_is_s3",
    "candidate_rank",
]


# ── Low-level similarity helpers ──────────────────────────────────────────────

def _jaccard_tokens(a: str, b: str) -> float:
    sa, sb = set(a.split()), set(b.split())
    if not sa and not sb:
        return 1.0
    if not sa or not sb:
        return 0.0
    return len(sa & sb) / len(sa | sb)


def _jaccard_numbers(a: str, b: str) -> float:
    na = set(re.findall(r"\d+", a))
    nb = set(re.findall(r"\d+", b))
    if not na and not nb:
        return 1.0
    if not na or not nb:
        return 0.0
    return len(na & nb) / len(na | nb)


def _len_ratio(a: str, b: str) -> float:
    la, lb = len(a), len(b)
    if la == 0 and lb == 0:
        return 1.0
    if la == 0 or lb == 0:
        return 0.0
    return min(la, lb) / max(la, lb)


def _extract_digit_set(text: str) -> set:
    if not text:
        return set()
    return set(re.findall(r"\d+", text))


if _HAS_RAPIDFUZZ:
    def _ratio(a, b):           return rfuzz.ratio(a, b) / 100.0
    def _token_sort(a, b):      return rfuzz.token_sort_ratio(a, b) / 100.0
    def _token_set(a, b):       return rfuzz.token_set_ratio(a, b) / 100.0
    def _partial(a, b):         return rfuzz.partial_ratio(a, b) / 100.0
    def _jaro_winkler(a, b):    return JaroWinkler.similarity(a, b)
else:
    logger.warning("rapidfuzz not available — using slower difflib fallback.")
    def _ratio(a, b):           return difflib.SequenceMatcher(None, a, b).ratio()
    def _token_sort(a, b):
        ta = " ".join(sorted(a.split()))
        tb = " ".join(sorted(b.split()))
        return difflib.SequenceMatcher(None, ta, tb).ratio()
    def _token_set(a, b):
        sa, sb = set(a.split()), set(b.split())
        inter = " ".join(sorted(sa & sb))
        only_a = " ".join(sorted(sa - sb))
        only_b = " ".join(sorted(sb - sa))
        best = max(
            difflib.SequenceMatcher(None, inter, inter + " " + only_a).ratio(),
            difflib.SequenceMatcher(None, inter, inter + " " + only_b).ratio(),
            difflib.SequenceMatcher(None, inter + " " + only_a, inter + " " + only_b).ratio(),
        )
        return best
    def _partial(a, b):
        short, long = (a, b) if len(a) <= len(b) else (b, a)
        best = 0.0
        for i in range(len(long) - len(short) + 1):
            r = difflib.SequenceMatcher(None, short, long[i:i+len(short)]).ratio()
            best = max(best, r)
        return best
    def _jaro_winkler(a, b):
        return difflib.SequenceMatcher(None, a, b).ratio()


# ── Feature Computation ───────────────────────────────────────────────────────

def compute_features_for_pair(
    s1_name: str, s1_addr: str, s1_country: str,
    s23_name: str, s23_addr: str, s23_country: str,
    s23_id: str = "",
    cand_rank: int = 1,
    blocking_score: float = 0.0,
) -> List[float]:
    """Compute the 28 features for a single (S1, S23) pair."""
    # Name features
    f_name_ratio       = _ratio(s1_name, s23_name)
    f_name_token_sort  = _token_sort(s1_name, s23_name)
    f_name_token_set   = _token_set(s1_name, s23_name)
    f_name_partial     = _partial(s1_name, s23_name)
    f_name_jw          = _jaro_winkler(s1_name, s23_name)
    f_name_jaccard     = _jaccard_tokens(s1_name, s23_name)

    # Address features
    f_addr_ratio       = _ratio(s1_addr, s23_addr)
    f_addr_token_sort  = _token_sort(s1_addr, s23_addr)
    f_addr_token_set   = _token_set(s1_addr, s23_addr)
    f_addr_jaccard     = _jaccard_tokens(s1_addr, s23_addr)
    f_addr_num_overlap = _jaccard_numbers(s1_addr, s23_addr)

    # Country & Blocking
    f_country_match    = float(s1_country == s23_country)
    f_blocking         = blocking_score

    # Structural
    f_name_len_s1      = float(len(s1_name))
    f_name_len_s23     = float(len(s23_name))
    f_name_len_ratio   = _len_ratio(s1_name, s23_name)
    f_addr_len_ratio   = _len_ratio(s1_addr, s23_addr)

    # Advanced Interactions & Domain features
    f_name_x_addr      = f_name_jw * f_addr_token_set
    f_min_name_addr    = min(f_name_token_set, f_addr_token_set)
    f_max_name_addr    = max(f_name_token_set, f_addr_token_set)

    l1, l2             = len(s1_addr), len(s23_addr)
    f_both_addr        = 1.0 if (l1 >= 5 and l2 >= 5) else 0.0
    f_missing_asym     = 1.0 if ((l1 >= 10 and l2 < 3) or (l2 >= 10 and l1 < 3)) else 0.0
    f_exact_name       = 1.0 if (s1_name and s1_name == s23_name) else 0.0

    d1, d2             = _extract_digit_set(s1_addr), _extract_digit_set(s23_addr)
    f_conflict_digits  = 1.0 if (d1 and d2 and not (d1 & d2)) else 0.0

    f_strong_name_num   = 1.0 if (f_name_token_set >= 0.85 and f_addr_num_overlap >= 0.5) else 0.0
    f_name_num_conflict = 1.0 if (f_name_token_set >= 0.85 and f_conflict_digits == 1.0) else 0.0

    f_is_s3            = 1.0 if s23_id.startswith("S3-") else 0.0
    f_cand_rank        = float(cand_rank)

    return [
        f_name_ratio, f_name_token_sort, f_name_token_set,
        f_name_partial, f_name_jw, f_name_jaccard,
        f_addr_ratio, f_addr_token_sort, f_addr_token_set,
        f_addr_jaccard, f_addr_num_overlap,
        f_country_match,
        f_blocking,
        f_name_len_s1, f_name_len_s23,
        f_name_len_ratio, f_addr_len_ratio,
        # Advanced 11
        f_name_x_addr, f_min_name_addr, f_max_name_addr,
        f_both_addr, f_missing_asym, f_exact_name,
        f_conflict_digits, f_strong_name_num, f_name_num_conflict,
        f_is_s3, f_cand_rank
    ]


def build_feature_matrix(
    pairs: List[Tuple[str, str]],
    s1_lookup: Dict,
    s23_lookup: Dict,
    blocking_scores: Dict[Tuple[str, str], float] = None,
    candidate_ranks: Dict[Tuple[str, str], int] = None,
    n_jobs: int = -1,
) -> np.ndarray:
    """Build feature matrix (N, 28) for pairs."""
    if blocking_scores is None:
        blocking_scores = {}
    if candidate_ranks is None:
        candidate_ranks = {}

    rows = []
    for s1_id, s23_id in pairs:
        s1  = s1_lookup[s1_id]
        s23 = s23_lookup[s23_id]
        b_score = blocking_scores.get((s1_id, s23_id), 0.0)
        c_rank  = candidate_ranks.get((s1_id, s23_id), 1)
        rows.append(compute_features_for_pair(
            s1["name_norm"],  s1["addr_norm"],  s1["country_norm"],
            s23["name_norm"], s23["addr_norm"], s23["country_norm"],
            s23_id=s23_id,
            cand_rank=c_rank,
            blocking_score=b_score,
        ))
    return np.array(rows, dtype=np.float32)


def build_lookup(df: pd.DataFrame) -> Dict:
    return {
        row["entity_id"]: {
            "name_norm":    row["name_norm"],
            "addr_norm":    row["addr_norm"],
            "country_norm": row["country_norm"],
        }
        for _, row in df.iterrows()
    }


In [ ]:
# --- evaluate.py ---
"""
evaluate.py — Local F₀.₅ scorer (matches the official leaderboard metric).

Usage
-----
    from evaluate import score_f05, f05_macro

    # from a results dict
    f05 = score_f05(pred_dict, gt_dict)

    # from TSV files
    f05 = score_from_files("output/matching_results.tsv",
                           "dataset/train/train_ground_truth.tsv")
"""

from pathlib import Path
from typing import Dict, List, Set

import pandas as pd


# ── Per-entity F₀.₅ ──────────────────────────────────────────────────────────

def f05_per_entity(
    predicted: Set[str],
    ground_truth: Set[str],
) -> float:
    """
    Compute F₀.₅ for a single Source-1 entity.

    • Singleton entity (no true matches):
        - Correct prediction (empty predicted) → 1.0
        - Any false merge → 0.0

    • Non-singleton entity:
        Standard precision-recall formula with β=0.5.
    """
    # Singleton case
    if not ground_truth:
        return 1.0 if not predicted else 0.0

    if not predicted:
        # missed all true matches
        precision, recall = 0.0, 0.0
    else:
        tp = len(predicted & ground_truth)
        precision = tp / len(predicted)
        recall    = tp / len(ground_truth)

    if precision == 0.0 and recall == 0.0:
        return 0.0

    beta_sq = 0.25  # β=0.5 → β²=0.25
    f05 = (1 + beta_sq) * precision * recall / (beta_sq * precision + recall)
    return f05


# ── Macro-averaged F₀.₅ ──────────────────────────────────────────────────────

def score_f05(
    pred_dict: Dict[str, List[str]],
    gt_dict:   Dict[str, List[str]],
    verbose:   bool = False,
) -> float:
    """
    Macro-average F₀.₅ across all S1 entities present in gt_dict.

    Parameters
    ----------
    pred_dict : {s1_id: [matched_s23_ids]} — your predictions
    gt_dict   : {s1_id: [matched_s23_ids]} — ground truth
    verbose   : print per-bucket stats

    Returns
    -------
    Macro-averaged F₀.₅ (float, 0-1)
    """
    scores = []
    singletons_correct = 0
    singletons_total   = 0
    non_single_scores  = []

    for s1_id, gt_ids in gt_dict.items():
        gt_set   = set(gt_ids) if gt_ids else set()
        pred_set = set(pred_dict.get(s1_id, []))
        f = f05_per_entity(pred_set, gt_set)
        scores.append(f)

        if not gt_set:
            singletons_total += 1
            if not pred_set:
                singletons_correct += 1
        else:
            non_single_scores.append(f)

    macro = sum(scores) / len(scores) if scores else 0.0

    if verbose:
        ns = len(non_single_scores)
        ns_avg = sum(non_single_scores) / ns if ns else 0.0
        print(f"  Entities evaluated : {len(scores)}")
        print(f"  Singletons         : {singletons_total} "
              f"(correctly empty: {singletons_correct})")
        print(f"  Non-singleton avg  : {ns_avg:.4f}  ({ns} entities)")
        print(f"  Macro F₀.₅         : {macro:.4f}")

    return macro


# ── From files ────────────────────────────────────────────────────────────────

def _load_results_tsv(path: Path) -> Dict[str, List[str]]:
    df = pd.read_csv(path, sep="\t", dtype=str, keep_default_na=False)
    result = {}
    for _, row in df.iterrows():
        s1_id = row["source1_entity_id"]
        ids_str = row.get("matched_entity_ids", "").strip()
        result[s1_id] = [x for x in ids_str.split(",") if x] if ids_str else []
    return result


def _load_gt_tsv(path: Path) -> Dict[str, List[str]]:
    return _load_results_tsv(path)


def score_from_files(
    pred_path: Path,
    gt_path:   Path,
    verbose:   bool = True,
) -> float:
    """Convenience wrapper: load TSV files and compute macro F₀.₅."""
    pred = _load_results_tsv(pred_path)
    gt   = _load_gt_tsv(gt_path)

    # Only score entities in the GT
    print(f"Scoring {len(gt)} GT entities vs {len(pred)} predicted...")
    return score_f05(pred, gt, verbose=verbose)


# ── CLI entry point ───────────────────────────────────────────────────────────

if __name__ == "__main__":
    import argparse
    parser = argparse.ArgumentParser(description="Compute local F₀.₅ score")
    parser.add_argument("--pred", required=True, help="Path to matching_results.tsv")
    parser.add_argument("--gt",   required=True, help="Path to ground truth TSV")
    args = parser.parse_args()

    f05 = score_from_files(Path(args.pred), Path(args.gt), verbose=True)
    print(f"\n>>> Macro F₀.₅ = {f05:.6f}")


In [ ]:
# --- experiment_runner.py ---
"""
experiment_runner.py — Experimentation, Feature Analysis, Error Diagnostics & Decision Policy Suite.

Runs:
1. 3-way S1 Entity Split: Train (70%), Val (15%), Holdout (15%).
2. Baseline LightGBM model training on candidate pairs.
3. Feature Importance Analysis (Gain, Split, Permutation Importance on Val).
4. Feature Ablation Group Tests.
5. Error Diagnostics (False Positives, False Negatives, Singletons, Over-merges).
6. Smart Entity-Level Decision Policies (Source-specific thresholds, Singleton confidence margin).
7. Structured logging to experiments/results.csv and reports/.
"""

import argparse
import logging
import os
import pickle
import time
from pathlib import Path
from typing import Dict, List, Tuple, Set

import numpy as np
import pandas as pd
import lightgbm as lgb

    EARLY_STOPPING_ROUNDS,
    K_CANDIDATES,
    LGBM_PARAMS,
    MODEL_DIR,
    NEG_PER_POS,
    OUTPUT_DIR,
    RANDOM_STATE,
    THRESHOLD_GRID,
    TRAIN_DIR,
    VAL_FRACTION,
)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-7s | %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger(__name__)

REPORTS_DIR = Path(__file__).resolve().parent.parent / "reports"
ERROR_DIR   = REPORTS_DIR / "error_analysis"
EXP_DIR     = Path(__file__).resolve().parent.parent / "experiments"

for d in (REPORTS_DIR, ERROR_DIR, EXP_DIR):
    d.mkdir(exist_ok=True)


# ── Data Loading & Split ──────────────────────────────────────────────────────

def load_and_preprocess_data(sample: int = 5000, full_s23: bool = False):
    cache_file = OUTPUT_DIR / f"preprocessed_cache_{sample}_{full_s23}.pkl"
    if cache_file.exists():
        logger.info(f"Loading preprocessed data from cache: {cache_file}")
        with open(cache_file, "rb") as f:
            return pickle.load(f)

    logger.info(f"Loading data (sample={sample})...")
    s1_raw = pd.read_csv(TRAIN_DIR / "train_source1.tsv", sep="\t", dtype=str)
    s2_raw = pd.read_csv(TRAIN_DIR / "train_source2.tsv", sep="\t", dtype=str)
    s3_raw = pd.read_csv(TRAIN_DIR / "train_source3.tsv", sep="\t", dtype=str)
    gt_raw = pd.read_csv(TRAIN_DIR / "train_ground_truth.tsv", sep="\t", dtype=str, keep_default_na=False)

    if sample:
        s1_raw = s1_raw.sample(n=min(sample, len(s1_raw)), random_state=RANDOM_STATE).reset_index(drop=True)
        sampled_ids = set(s1_raw["entity_id"])
        gt_subset   = gt_raw[gt_raw["source1_entity_id"].isin(sampled_ids)]
        gt_match_ids: set = set()
        for _, row in gt_subset.iterrows():
            ids_str = row.get("matched_entity_ids", "").strip()
            if ids_str:
                gt_match_ids.update(ids_str.split(","))

        gt_s2_ids = {x for x in gt_match_ids if x.startswith("S2-")}
        gt_s3_ids = {x for x in gt_match_ids if x.startswith("S3-")}

        rng = np.random.default_rng(RANDOM_STATE)
        def _subsample(df, gt_ids, n_extra):
            gt_rows = df[df["entity_id"].isin(gt_ids)]
            rest    = df[~df["entity_id"].isin(gt_ids)]
            n_take  = min(n_extra, len(rest))
            extra   = rest.iloc[rng.choice(len(rest), n_take, replace=False)]
            return pd.concat([gt_rows, extra], ignore_index=True)

        if not full_s23:
            s2_raw = _subsample(s2_raw, gt_s2_ids, sample * 5)
            s3_raw = _subsample(s3_raw, gt_s3_ids, sample * 5)
        else:
            logger.info("Keeping FULL S2/S3 (Realistic blocking test)")

    s1 = preprocess_df(s1_raw)
    s2 = preprocess_df(s2_raw)
    s3 = preprocess_df(s3_raw)

    gt_dict = {}
    s1_id_set = set(s1["entity_id"])
    for _, row in gt_raw.iterrows():
        sid = row["source1_entity_id"]
        if sid in s1_id_set:
            m = row.get("matched_entity_ids", "").strip()
            gt_dict[sid] = [x for x in m.split(",") if x] if m else []

    res = (s1, s2, s3, gt_dict)
    with open(cache_file, "wb") as f:
        pickle.dump(res, f)
    return res


def split_three_way(s1: pd.DataFrame, train_frac=0.70, val_frac=0.15):
    """Deterministic 3-way split: Train (70%), Val (15%), Holdout (15%) by S1 entity."""
    s1 = s1.copy()
    rng = np.random.default_rng(RANDOM_STATE)
    
    val_ids, holdout_ids = set(), set()
    for country, grp in s1.groupby("country_norm"):
        n_grp = len(grp)
        n_val = max(1, int(n_grp * val_frac))
        n_hold = max(1, int(n_grp * val_frac))
        
        shuffled_eids = grp.sample(frac=1.0, random_state=RANDOM_STATE)["entity_id"].tolist()
        val_ids.update(shuffled_eids[:n_val])
        holdout_ids.update(shuffled_eids[n_val:n_val+n_hold])

    train_mask = ~s1["entity_id"].isin(val_ids | holdout_ids)
    val_mask   = s1["entity_id"].isin(val_ids)
    hold_mask  = s1["entity_id"].isin(holdout_ids)

    s1_train   = s1[train_mask].copy().reset_index(drop=True)
    s1_val     = s1[val_mask].copy().reset_index(drop=True)
    s1_holdout = s1[hold_mask].copy().reset_index(drop=True)

    return s1_train, s1_val, s1_holdout


def build_pairs_with_missed(s1_df, cands, gt_dict, neg_per_pos=NEG_PER_POS):
    rng = np.random.default_rng(RANDOM_STATE)
    pairs, labels = [], []
    for sid in s1_df["entity_id"]:
        cand_list = cands.get(sid, [])
        gt_set    = set(gt_dict.get(sid, []))

        pos = [c for c in cand_list if c in gt_set]
        neg = [c for c in cand_list if c not in gt_set]
        missed = [g for g in gt_set if g not in cand_list]
        if missed:
            pos.extend(missed)

        for c in pos:
            pairs.append((sid, c))
            labels.append(1)

        n_neg = min(len(neg), max(neg_per_pos * len(pos), 1))
        sampled_neg = rng.choice(neg, size=n_neg, replace=False).tolist() if neg else []
        for c in sampled_neg:
            pairs.append((sid, c))
            labels.append(0)

    return pairs, labels


# ── Feature Evaluation & Permutation Importance ────────────────────────────────

def compute_permutation_importance(model, X_val, y_val, feature_names, scoring_metric="auc"):
    from sklearn.metrics import roc_auc_score
    baseline_probs = model.predict(X_val)
    baseline_score = roc_auc_score(y_val, baseline_probs)
    
    importances = {}
    rng = np.random.default_rng(RANDOM_STATE)
    for i, col in enumerate(feature_names):
        X_perm = X_val.copy()
        X_perm[:, i] = rng.permutation(X_perm[:, i])
        perm_probs = model.predict(X_perm)
        perm_score = roc_auc_score(y_val, perm_probs)
        importances[col] = baseline_score - perm_score
    return importances


# ── Detailed Metrics & Diagnostics ────────────────────────────────────────────

def evaluate_predictions_detailed(pred_dict: Dict[str, List[str]], gt_dict: Dict[str, List[str]]):
    total_entities = len(gt_dict)
    singleton_total = 0
    singleton_correct = 0
    
    total_tp = 0
    total_fp = 0
    total_fn = 0
    
    scores = []
    
    for sid, gt_ids in gt_dict.items():
        gt_set   = set(gt_ids) if gt_ids else set()
        pred_set = set(pred_dict.get(sid, []))
        
        f05 = f05_per_entity(pred_set, gt_set)
        scores.append(f05)
        
        if not gt_set:
            singleton_total += 1
            if not pred_set:
                singleton_correct += 1
            else:
                total_fp += len(pred_set)
        else:
            tp = len(pred_set & gt_set)
            fp = len(pred_set - gt_set)
            fn = len(gt_set - pred_set)
            total_tp += tp
            total_fp += fp
            total_fn += fn

    macro_f05 = sum(scores) / len(scores) if scores else 0.0
    precision = total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0.0
    recall    = total_tp / (total_tp + total_fn) if (total_tp + total_fn) > 0 else 0.0
    singleton_acc = singleton_correct / singleton_total if singleton_total > 0 else 1.0

    return {
        "macro_f05": macro_f05,
        "precision": precision,
        "recall": recall,
        "singleton_acc": singleton_acc,
        "tp": total_tp,
        "fp": total_fp,
        "fn": total_fn,
        "num_pred_links": total_tp + total_fp,
    }


# ── Logging to CSV ─────────────────────────────────────────────────────────────

def log_experiment_result(result_dict: dict):
    results_path = EXP_DIR / "results.csv"
    df_new = pd.DataFrame([result_dict])
    if results_path.exists():
        df_old = pd.read_csv(results_path)
        df_combined = pd.concat([df_old, df_new], ignore_index=True)
    else:
        df_combined = df_new
    df_combined.to_csv(results_path, index=False)
    logger.info(f"Experiment result logged to {results_path}")


# ── Run Baseline Audit Experiment ──────────────────────────────────────────────

def run_audit_experiment(sample=5000, k_candidates=50, full_s23=False):
    logger.info("=== Running Step 1 & 2 Audit & Baseline Experiment ===")
    t0 = time.time()
    
    s1, s2, s3, gt_dict = load_and_preprocess_data(sample=sample, full_s23=full_s23)
    s1_train, s1_val, s1_holdout = split_three_way(s1)
    
    logger.info(f"Splits: Train={len(s1_train)}, Val={len(s1_val)}, Holdout={len(s1_holdout)}")
    
    s23_lookup = build_lookup(pd.concat([s2, s3], ignore_index=True))
    s1_train_lookup = build_lookup(s1_train)
    s1_val_lookup   = build_lookup(s1_val)
    s1_hold_lookup  = build_lookup(s1_holdout)

    # 1. Blocking
    logger.info("Running candidate blocking...")
    cands_cache_file = OUTPUT_DIR / f"cands_cache_{sample}_{full_s23}_{k_candidates}.pkl"
    if cands_cache_file.exists():
        logger.info(f"Loading candidates from cache: {cands_cache_file}")
        with open(cands_cache_file, "rb") as f:
            cands_train, cands_val, cands_hold = pickle.load(f)
    else:
        cands_train = generate_candidates(s1_train, s2, s3, k=k_candidates)
        cands_val   = generate_candidates(s1_val, s2, s3, k=k_candidates)
        cands_hold  = generate_candidates(s1_holdout, s2, s3, k=k_candidates)
        with open(cands_cache_file, "wb") as f:
            pickle.dump((cands_train, cands_val, cands_hold), f)

    gt_train = {k: v for k, v in gt_dict.items() if k in set(s1_train["entity_id"])}
    gt_val   = {k: v for k, v in gt_dict.items() if k in set(s1_val["entity_id"])}
    gt_hold  = {k: v for k, v in gt_dict.items() if k in set(s1_holdout["entity_id"])}

    # 2. Build pairs & features
    train_pairs, train_labels = build_pairs_with_missed(s1_train, cands_train, gt_train)
    val_pairs, val_labels     = build_pairs_with_missed(s1_val, cands_val, gt_val)
    hold_pairs, hold_labels   = build_pairs_with_missed(s1_holdout, cands_hold, gt_hold)

    feat_cache_file = OUTPUT_DIR / f"features_cache_{sample}_{full_s23}_{k_candidates}.pkl"
    if feat_cache_file.exists():
        logger.info(f"Loading features from cache: {feat_cache_file}")
        with open(feat_cache_file, "rb") as f:
            X_train, y_train, X_val, y_val, X_hold, y_hold = pickle.load(f)
    else:
        X_train = build_feature_matrix(train_pairs, s1_train_lookup, s23_lookup, n_jobs=-1)
        y_train = np.array(train_labels, dtype=np.int32)

        X_val = build_feature_matrix(val_pairs, s1_val_lookup, s23_lookup, n_jobs=-1)
        y_val = np.array(val_labels, dtype=np.int32)

        X_hold = build_feature_matrix(hold_pairs, s1_hold_lookup, s23_lookup, n_jobs=-1)
        y_hold = np.array(hold_labels, dtype=np.int32)
        
        with open(feat_cache_file, "wb") as f:
            pickle.dump((X_train, y_train, X_val, y_val, X_hold, y_hold), f)

    # 3. Train LightGBM
    dtrain = lgb.Dataset(X_train, label=y_train, feature_name=FEATURE_NAMES)
    dval   = lgb.Dataset(X_val, label=y_val, feature_name=FEATURE_NAMES, reference=dtrain)

    params = dict(LGBM_PARAMS)
    params.pop("n_estimators", None)
    model = lgb.train(
        params,
        dtrain,
        num_boost_round=LGBM_PARAMS["n_estimators"],
        valid_sets=[dval],
        callbacks=[
            lgb.early_stopping(EARLY_STOPPING_ROUNDS, verbose=False),
        ],
    )

    # 4. Feature Importances
    gain_imp  = model.feature_importance(importance_type="gain")
    split_imp = model.feature_importance(importance_type="split")
    perm_imp  = compute_permutation_importance(model, X_val, y_val, FEATURE_NAMES)

    fi_df = pd.DataFrame({
        "feature": FEATURE_NAMES,
        "gain_importance": gain_imp,
        "split_importance": split_imp,
        "val_perm_importance": [perm_imp[f] for f in FEATURE_NAMES]
    }).sort_values(by="gain_importance", ascending=False)
    
    fi_df.to_csv(REPORTS_DIR / "feature_importance.csv", index=False)
    logger.info(f"Saved feature importance report to {REPORTS_DIR / 'feature_importance.csv'}")

    # Feature Correlations
    X_val_df = pd.DataFrame(X_val, columns=FEATURE_NAMES)
    corr_df  = X_val_df.corr()
    corr_df.to_csv(REPORTS_DIR / "feature_correlations.csv")

    # 5. Threshold search on validation
    val_probs = model.predict(X_val)
    prob_map_val = {(s1_id, s23_id): prob for (s1_id, s23_id), prob in zip(val_pairs, val_probs)}
    
    best_val_f05 = -1.0
    best_thr = 0.5
    best_val_stats = {}

    for thr in THRESHOLD_GRID:
        pred_dict_val = {}
        for sid in s1_val["entity_id"]:
            cands = cands_val.get(sid, [])
            matched = [c for c in cands if prob_map_val.get((sid, c), 0.0) >= thr]
            pred_dict_val[sid] = matched

        stats = evaluate_predictions_detailed(pred_dict_val, gt_val)
        if stats["macro_f05"] > best_val_f05:
            best_val_f05 = stats["macro_f05"]
            best_thr = thr
            best_val_stats = stats

    logger.info(f"★ Best Global Threshold on Val: {best_thr:.2f} | Val F0.5: {best_val_f05:.4f}")

    # 6. Evaluate Holdout using best threshold
    hold_probs = model.predict(X_hold)
    prob_map_hold = {(s1_id, s23_id): prob for (s1_id, s23_id), prob in zip(hold_pairs, hold_probs)}
    pred_dict_hold = {}
    for sid in s1_holdout["entity_id"]:
        cands = cands_hold.get(sid, [])
        matched = [c for c in cands if prob_map_hold.get((sid, c), 0.0) >= best_thr]
        pred_dict_hold[sid] = matched
    
    hold_stats = evaluate_predictions_detailed(pred_dict_hold, gt_hold)
    logger.info(f"★ Holdout F0.5 (thr={best_thr:.2f}): {hold_stats['macro_f05']:.4f}")

    # 7. Error Diagnostics Export (Val set)
    export_error_diagnostics(s1_val, s23_lookup, cands_val, gt_val, prob_map_val, best_thr)

    # 8. Log experiment
    exp_record = {
        "experiment_id": "EXP-001-BASELINE-AUDIT",
        "timestamp": pd.Timestamp.now().isoformat(),
        "git_commit_if_available": "clean",
        "candidate_K": k_candidates,
        "feature_set": "baseline_17",
        "added_features": "none",
        "removed_features": "none",
        "model_parameters": f"num_leaves={LGBM_PARAMS['num_leaves']},lr={LGBM_PARAMS['learning_rate']}",
        "negative_sampling_strategy": f"hard_neg_ratio={NEG_PER_POS}",
        "threshold_policy": "global_grid_search",
        "global_threshold": best_thr,
        "S2_threshold": best_thr,
        "S3_threshold": best_thr,
        "validation_precision": round(best_val_stats["precision"], 4),
        "validation_recall": round(best_val_stats["recall"], 4),
        "validation_f05": round(best_val_f05, 4),
        "validation_singleton_accuracy": round(best_val_stats["singleton_acc"], 4),
        "holdout_f05_if_run": round(hold_stats["macro_f05"], 4),
        "num_predictions": best_val_stats["num_pred_links"],
        "false_positives": best_val_stats["fp"],
        "false_negatives": best_val_stats["fn"],
        "runtime_seconds": round(time.time() - t0, 1),
        "notes": "Step 1 audit baseline run on 5K sample with 3-way split (Train/Val/Holdout)"
    }
    log_experiment_result(exp_record)

    # Save top_features.md summary
    save_top_features_report(fi_df)
    
    model_path = MODEL_DIR / "lgbm_model.pkl"
    with open(model_path, "wb") as f:
        pickle.dump((model, best_thr), f)
    logger.info(f"Saved final trained model to {model_path}")
    
    return model, best_thr, best_val_f05, hold_stats["macro_f05"]


def export_error_diagnostics(s1_df, s23_lookup, cands_dict, gt_dict, prob_map, threshold):
    fps, fns, singletons_fm, over_merges = [], [], [], []

    s1_lookup = build_lookup(s1_df)

    for sid in s1_df["entity_id"]:
        gt_set = set(gt_dict.get(sid, []))
        cands  = cands_dict.get(sid, [])
        pred_set = set([c for c in cands if prob_map.get((sid, c), 0.0) >= threshold])

        s1_info = s1_lookup[sid]

        # Singletons False Merges
        if not gt_set and pred_set:
            for p in pred_set:
                s23_info = s23_lookup[p]
                singletons_fm.append({
                    "s1_id": sid,
                    "cand_id": p,
                    "source": "S2" if p.startswith("S2-") else "S3",
                    "s1_name": s1_info["name_norm"],
                    "s23_name": s23_info["name_norm"],
                    "s1_addr": s1_info["addr_norm"],
                    "s23_addr": s23_info["addr_norm"],
                    "country": s1_info["country_norm"],
                    "prob": round(prob_map.get((sid, p), 0.0), 4),
                })

        # False Positives on non-singletons
        if gt_set:
            for p in (pred_set - gt_set):
                s23_info = s23_lookup[p]
                fps.append({
                    "s1_id": sid,
                    "cand_id": p,
                    "source": "S2" if p.startswith("S2-") else "S3",
                    "s1_name": s1_info["name_norm"],
                    "s23_name": s23_info["name_norm"],
                    "s1_addr": s1_info["addr_norm"],
                    "s23_addr": s23_info["addr_norm"],
                    "country": s1_info["country_norm"],
                    "prob": round(prob_map.get((sid, p), 0.0), 4),
                })

        # False Negatives (Candidate existed & was GT match, but model rejected)
        for g in gt_set:
            if g in cands and g not in pred_set:
                s23_info = s23_lookup[g]
                fns.append({
                    "s1_id": sid,
                    "cand_id": g,
                    "source": "S2" if g.startswith("S2-") else "S3",
                    "s1_name": s1_info["name_norm"],
                    "s23_name": s23_info["name_norm"],
                    "s1_addr": s1_info["addr_norm"],
                    "s23_addr": s23_info["addr_norm"],
                    "country": s1_info["country_norm"],
                    "prob": round(prob_map.get((sid, g), 0.0), 4),
                })

    pd.DataFrame(fps).to_csv(ERROR_DIR / "false_positives.csv", index=False)
    pd.DataFrame(fns).to_csv(ERROR_DIR / "false_negatives.csv", index=False)
    pd.DataFrame(singletons_fm).to_csv(ERROR_DIR / "singleton_false_merges.csv", index=False)
    logger.info(f"Exported error CSVs (FP: {len(fps)}, FN: {len(fns)}, Singleton FM: {len(singletons_fm)}) to {ERROR_DIR}")


def run_predict_experiment():
    logger.info("=== Running Final Inference on Test Set ===")
    from config import TEST_DIR, K_CANDIDATES
    
    # Load test data
    s1_raw = pd.read_csv(TEST_DIR / "test_source1.tsv", sep="	", dtype=str)
    s2_raw = pd.read_csv(TEST_DIR / "test_source2.tsv", sep="	", dtype=str)
    s3_raw = pd.read_csv(TEST_DIR / "test_source3.tsv", sep="	", dtype=str)
    
    s1 = preprocess_df(s1_raw)
    s2 = preprocess_df(s2_raw)
    s3 = preprocess_df(s3_raw)
    
    cands_cache_file = OUTPUT_DIR / f"cands_test.pkl"
    if cands_cache_file.exists():
        with open(cands_cache_file, "rb") as f:
            cands_test = pickle.load(f)
    else:
        cands_test = generate_candidates(s1, s2, s3, k=K_CANDIDATES)
        with open(cands_cache_file, "wb") as f:
            pickle.dump(cands_test, f)
            
    # Build features
    s1_lookup = build_lookup(s1)
    s23_lookup = build_lookup(pd.concat([s2, s3], ignore_index=True))
    
    pairs = []
    for sid in s1["entity_id"]:
        for c in cands_test.get(sid, []):
            pairs.append((sid, c))
            
    feat_cache = OUTPUT_DIR / "features_test.pkl"
    if feat_cache.exists():
        with open(feat_cache, "rb") as f:
            X_test = pickle.load(f)
    else:
        X_test = build_feature_matrix(pairs, s1_lookup, s23_lookup, n_jobs=-1)
        with open(feat_cache, "wb") as f:
            pickle.dump(X_test, f)
            
    # Load model
    model_path = MODEL_DIR / "lgbm_model.pkl"
    if not model_path.exists():
        raise FileNotFoundError("Model not found. Run 'train' first.")
    with open(model_path, "rb") as f:
        model, best_thr = pickle.load(f)
        
    probs = model.predict(X_test)
    prob_map = {(sid, cid): prob for (sid, cid), prob in zip(pairs, probs)}
    
    submission_rows = []
    for sid in s1["entity_id"]:
        cands = cands_test.get(sid, [])
        matched = [c for c in cands if prob_map.get((sid, c), 0.0) >= best_thr]
        submission_rows.append({
            "source1_entity_id": sid,
            "target_entity_ids": ",".join(matched)
        })
        
    sub_df = pd.DataFrame(submission_rows)
    sub_df.to_csv(OUTPUT_DIR / "submission.csv", index=False)
    logger.info(f"Saved submission to {OUTPUT_DIR / 'submission.csv'}")

def save_top_features_report(fi_df: pd.DataFrame):
    lines = [
        "# Top Features Ranking & Importance Analysis",
        "",
        "Ranked by LightGBM **Gain Importance** on Validation Set.",
        "",
        "| Rank | Feature Name | Gain Importance | Split Count | Val Permutation Importance |",
        "|------|--------------|-----------------|-------------|----------------------------|",
    ]
    for idx, row in fi_df.reset_index(drop=True).iterrows():
        lines.append(
            f"| {idx+1} | `{row['feature']}` | {row['gain_importance']:.2f} | {int(row['split_importance'])} | {row['val_perm_importance']:.6f} |"
        )
    with open(REPORTS_DIR / "top_features.md", "w", encoding="utf-8") as f:
        f.write("\n".join(lines))
    logger.info(f"Saved {REPORTS_DIR / 'top_features.md'}")


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("mode", choices=["train", "predict"], nargs="?", default="train")
    parser.add_argument("--sample", type=int, default=0, help="0 means full data")
    parser.add_argument("--full-s23", action="store_true", help="Do not sample S2/S3 (Realistic blocking test)")
    args = parser.parse_args()
    
    if args.mode == "train":
        if args.sample == 50000 and args.full_s23:
            logger.info("CRITICAL CHECKPOINT: Running 50K S1 against FULL 10M S2/S3")
        elif args.sample == 0:
            logger.info("CRITICAL CHECKPOINT: Running FULL PIPELINE on 2.2M rows")
        run_audit_experiment(sample=args.sample, full_s23=args.full_s23)
    elif args.mode == "predict":
        run_predict_experiment()


In [ ]:
# RUN FULL TRAINING
# Set sample=0 to run on the full 2.2M dataset!
# Using sample=50000 and full_s23=True for the realistic scalable test.
run_audit_experiment(sample=50000, full_s23=True)


In [ ]:
# RUN INFERENCE ON TEST SET
run_predict_experiment()
